# 05 · Open Question

## Question: Does being a multilingual, non-Anglophone actor correlate with higher IMDb ratings or longer careers?

**Why is this interesting?**
Hollywood is often criticized for its Anglo-centric bias. Conversely, international actors who successfully cross over into English-language cinema (or work across multiple film industries) might be perceived as exceptionally talented. By combining IMDb data (ratings, roles) with enriched Wikipedia biographies (country of birth) and Wikidata (original film languages), we can quantify whether multilingual non-Anglophone actors achieve better average ratings or longer careers than monolingual Anglophone actors.

**Methodology:**
1. Load the cleaned IMDb tables (principals, titles, ratings) and the enriched Wikipedia biographies.
2. Extract the country of origin from the Wikipedia biography text using regex heuristics.
3. Join the actors with their films and the original languages of those films.
4. Classify actors into groups: Anglophone vs. Non-Anglophone, and Monolingual vs. Multilingual.
5. Compare average IMDb ratings and career length.

In [7]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from the IMDb repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType

from src.imdb_loader import data_root, prepared_snapshot, read_prepared, spark_session

DATA = data_root(ROOT)
snapshot = prepared_snapshot(DATA)
spark = spark_session("IMDb Open Question")
spark.sparkContext.setLogLevel("WARN")

tables = read_prepared(spark, snapshot)

# Identify the enrichment output directory for the current snapshot
enrich_dir = DATA / "enrichment" / "output" / snapshot.name
if not enrich_dir.is_dir():
    raise FileNotFoundError(f"Enrichment output not found at {enrich_dir}. Run 03_data_enrichment.ipynb first.")

# Load enriched principals (containing Wikipedia biographies) and original languages
principals_enriched = spark.read.parquet((enrich_dir / "principals").as_posix())
languages = spark.read.csv((DATA / "reference" / "original_languages.csv").as_posix(), header=True, inferSchema=True)

print(f"Enriched principals loaded: {principals_enriched.count():,} rows")
print(f"Languages loaded: {languages.count():,} rows")

Enriched principals loaded: 21,493 rows
Languages loaded: 7,403 rows


## Step 1: Extract country of birth from Wikipedia biographies

The Wikipedia biography usually starts with a sentence like:
*"James Maitland Stewart (May 20, 1908 - July 2, 1997) was an American actor..."*
*"José Antonio Domínguez Bandera (born 10 August 1960), known as Antonio Banderas, is a Spanish actor..."*

We map the most common nationality adjectives ("American", "British", "French"...) found in the first 300 characters of the biography to a country.

In [8]:
# Map nationality adjectives to country names
NATIONALITY_MAP = {
    "American": "United States", "British": "United Kingdom", "English": "United Kingdom",
    "Scottish": "United Kingdom", "Welsh": "United Kingdom", "Irish": "Ireland",
    "Canadian": "Canada", "Australian": "Australia", "French": "France",
    "German": "Germany", "Italian": "Italy", "Spanish": "Spain",
    "Mexican": "Mexico", "Japanese": "Japan", "Chinese": "China",
    "Indian": "India", "Russian": "Russia", "Swedish": "Sweden",
    "Norwegian": "Norway", "Danish": "Denmark", "Dutch": "Netherlands",
    "Polish": "Poland", "Brazilian": "Brazil", "Argentine": "Argentina",
}

# Deduplicate actors and keep the first 300 characters of the bio
actors_base = principals_enriched.select(
    "nconst", "primaryName", "biography"
).dropDuplicates(["nconst"]).withColumn(
    "bio_snippet", F.substring("biography", 1, 300)
)

# Build a native Spark expression (no Python UDF, avoids worker crashes)
country_expr = F.lit("Unknown")
for adj, country in NATIONALITY_MAP.items():
    country_expr = F.when(
        F.col("bio_snippet").contains(f" {adj} "), F.lit(country)
    ).otherwise(country_expr)

actors_with_country = actors_base.withColumn("country", country_expr)

print("Distribution of extracted countries:")
actors_with_country.groupBy("country").count().orderBy(F.desc("count")).show(15, truncate=False)

Distribution of extracted countries:
+--------------+-----+
|country       |count|
+--------------+-----+
|Unknown       |4177 |
|United States |3583 |
|United Kingdom|886  |
|India         |522  |
|Japan         |305  |
|France        |305  |
|Canada        |267  |
|Australia     |174  |
|Italy         |150  |
|Germany       |130  |
|Spain         |124  |
|Russia        |107  |
|Ireland       |94   |
|Sweden        |72   |
|Mexico        |64   |
+--------------+-----+
only showing top 15 rows



## Step 2: Prepare the actor-career dataset

We now build a dataset with one row per actor, containing:
- Their extracted country.
- Whether they are Anglophone.
- The number of distinct original languages of their films.
- Their average IMDb rating (only for films with at least 100 votes).
- Their career length (last film year - first film year).

In [9]:
# 1. Extract films and ratings (keep >= 100 votes to reduce noise)
films = tables["title.basics"].filter(F.col("titleType") == "movie").select("tconst", "startYear")
ratings = tables["title.ratings"].select("tconst", "averageRating", "numVotes")
films_rated = films.join(ratings, "tconst")

# 2. Get acting credits only
actors_credits = tables["title.principals"].filter(
    F.col("category").isin("actor", "actress")
).select("tconst", "nconst").distinct()

actor_films = actors_credits.join(films_rated, "tconst")

# 3. Join with languages (left join to keep films without a documented language)
actor_languages = actor_films.join(languages, "tconst", "left")

# 4. Aggregate metrics per actor
actor_stats = actor_languages.groupBy("nconst").agg(
    F.countDistinct("language").alias("language_count"),
    F.avg(F.when(F.col("numVotes") >= 100, F.col("averageRating"))).alias("avg_rating"),
    F.min("startYear").alias("first_year"),
    F.max("startYear").alias("last_year"),
    F.count("*").alias("film_count")
).withColumn(
    "career_length", F.col("last_year") - F.col("first_year")
)

# 5. Join with the country extracted from the biography
final_df = actor_stats.join(actors_with_country.select("nconst", "primaryName", "country"), "nconst")

# 6. Create categorical columns for the analysis
anglophone_countries = ["United States", "United Kingdom", "Canada", "Australia", "Ireland"]

final_df = final_df.withColumn(
    "is_anglophone", 
    F.when(F.col("country").isin(anglophone_countries), "Anglophone")
     .when(F.col("country") == "Unknown", "Unknown")
     .otherwise("Non-Anglophone")
).withColumn(
    "is_multilingual",
    F.when(F.col("language_count") >= 2, "Multilingual")
     .otherwise("Monolingual")
)

print(f"Actors analyzed (with bio and films): {final_df.count():,}")

Actors analyzed (with bio and films): 9,993


## Step 3: Results, Multilingualism and ratings

We compare average IMDb rating and career length between multilingual and monolingual actors. We keep actors with at least 5 films and a known country to avoid noise.

In [10]:
# Filter for actors with a significant career and known country
clean_actors = final_df.filter(
    (F.col("film_count") >= 5) & 
    (F.col("country") != "Unknown") & 
    (F.col("avg_rating").isNotNull())
)

print("=== Impact of multilingualism (all countries) ===")
clean_actors.groupBy("is_multilingual").agg(
    F.count("*").alias("num_actors"),
    F.round(F.avg("avg_rating"), 2).alias("avg_imdb_rating"),
    F.round(F.avg("career_length"), 1).alias("avg_career_years"),
    F.round(F.avg("film_count"), 1).alias("avg_films")
).orderBy("is_multilingual").show(truncate=False)

print("\n=== Cross-tab: Anglophone/Non-Anglophone x Multilingual ===")
clean_actors.filter(F.col("is_anglophone") != "Unknown").groupBy("is_anglophone", "is_multilingual").agg(
    F.count("*").alias("num_actors"),
    F.round(F.avg("avg_rating"), 2).alias("avg_imdb_rating"),
    F.round(F.avg("career_length"), 1).alias("avg_career_years")
).orderBy("is_anglophone", "is_multilingual").show(truncate=False)

=== Impact of multilingualism (all countries) ===


+---------------+----------+---------------+----------------+---------+
|is_multilingual|num_actors|avg_imdb_rating|avg_career_years|avg_films|
+---------------+----------+---------------+----------------+---------+
|Monolingual    |4010      |5.88           |28.9            |27.3     |
|Multilingual   |1741      |6.14           |33.3            |44.7     |
+---------------+----------+---------------+----------------+---------+


=== Cross-tab: Anglophone/Non-Anglophone x Multilingual ===
+--------------+---------------+----------+---------------+----------------+
|is_anglophone |is_multilingual|num_actors|avg_imdb_rating|avg_career_years|
+--------------+---------------+----------+---------------+----------------+
|Anglophone    |Monolingual    |2840      |5.76           |28.0            |
|Anglophone    |Multilingual   |1066      |6.11           |32.5            |
|Non-Anglophone|Monolingual    |1170      |6.17           |31.2            |
|Non-Anglophone|Multilingual   |675       |6

## Step 4: Top countries by average rating

We look at the countries with the highest average IMDb ratings (actors with at least 5 films).

In [11]:
print("Top 10 countries by average IMDb rating: ")
clean_actors.groupBy("country").agg(
    F.count("*").alias("num_actors"),
    F.round(F.avg("avg_rating"), 2).alias("avg_imdb_rating"),
    F.round(F.avg("career_length"), 1).alias("avg_career_years")
).filter(F.col("num_actors") >= 10).orderBy(F.desc("avg_imdb_rating")).show(10, truncate=False)

Top 10 countries by average IMDb rating: 
+--------------+----------+---------------+----------------+
|country       |num_actors|avg_imdb_rating|avg_career_years|
+--------------+----------+---------------+----------------+
|Japan         |270       |6.66           |30.0            |
|Germany       |114       |6.31           |31.7            |
|Brazil        |42        |6.3            |34.3            |
|Argentina     |26        |6.26           |35.3            |
|Ireland       |71        |6.19           |26.9            |
|Russia        |93        |6.17           |33.9            |
|China         |56        |6.17           |29.1            |
|Norway        |29        |6.16           |31.7            |
|India         |490       |6.13           |28.9            |
|United Kingdom|699       |6.12           |28.5            |
+--------------+----------+---------------+----------------+
only showing top 10 rows



## Key Takeaways

- **Multilingual actors** have a higher average IMDb rating (6.14) compared to monolingual actors (5.88). They also have significantly longer careers (33.3 years vs 28.9 years) and appear in more films on average (44.7 vs 27.3).
- **Anglophone vs. Non-Anglophone**: Non-Anglophone actors generally achieve higher average ratings than Anglophone actors, regardless of whether they are monolingual or multilingual.
- **Top Countries**: Actors born in Japan have the highest average IMDb rating (6.66), followed by Germany and Brazil.
- **Limitations**: The country extraction relies on a heuristic on the first 300 characters of the biography. Some actors may be misclassified. The multilingualism proxy uses the original language of the films, not the actor's actual spoken languages.